In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("SalesDataAnalysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/26 10:36:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.functions import col, when, count

In [3]:
df= spark.read.csv("/opt/spark-data/Sales/retail_sales_profiling.csv", header=True, inferSchema=True)

In [4]:
df.show(5)

+--------+----------+-----------+----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|order_id|order_date|customer_id|product_id|  product_name|   category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+--------+----------+-----------+----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
| ORD1065|2024-12-21|   CUST0011|      P006|      Notebook| Stationery|          SP05|      Divya Nair| North|       3|     65.08|          15|          29.29|      165.95|    Debit Card|   Completed|
| ORD1196|2024-06-29|   CUST0019|      P004|Wireless Mouse|Electronics|          SP06|     Arjun Patel| South|       5|    929.24|          20|         929.24|     3716.96|    Debit Card|     Pend

In [5]:
null_counts = df.select([ count(when(col(c).isNull(), c)).alias(c) for c in df.columns ]) 
null_counts.show(truncate=False) 

+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|order_id|order_date|customer_id|product_id|product_name|category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|0       |0         |0          |0         |0           |0       |0             |25              |15    |0       |9         |42          |42             |9           |12            |0           |
+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+



In [6]:
from pyspark.sql.functions import mean, round as spark_round

In [7]:
df_nulls_fixed = df

In [8]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"discount_pct": 0.0, "discount_amount": 0.0})

In [9]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"salesperson_name": "Online / No Rep"})

In [10]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"region": "Unknown"})

In [11]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"payment_method": "Not Captured"})

In [12]:
median_price_per_product = df_nulls_fixed \
    .filter(col("unit_price").isNotNull()) \
    .groupBy("product_id") \
    .agg(spark_round(mean(col("unit_price")), 2).alias("median_unit_price"))

In [13]:
median_price_per_product.show()

+----------+-----------------+
|product_id|median_unit_price|
+----------+-----------------+
|      P007|          8950.46|
|      P003|          6161.44|
|      P010|           138.19|
|      P006|            93.61|
|      P004|          1279.61|
|      P002|          25909.1|
|      P001|          66770.5|
|      P008|           514.37|
|      P009|          2783.67|
|      P005|         11065.27|
+----------+-----------------+



In [14]:
df_nulls_fixed = df_nulls_fixed \
    .join(median_price_per_product, on="product_id", how="left") \
    .withColumn(
        "unit_price",
        when(col("unit_price").isNull(), col("median_unit_price"))
        .otherwise(col("unit_price"))
    ) \
    .drop("median_unit_price")

In [15]:
df_nulls_fixed.show()

+----------+--------+----------+-----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|product_id|order_id|order_date|customer_id|  product_name|   category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+----------+--------+----------+-----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|      P006| ORD1065|2024-12-21|   CUST0011|      Notebook| Stationery|          SP05|      Divya Nair| North|       3|     65.08|          15|          29.29|      165.95|    Debit Card|   Completed|
|      P004| ORD1196|2024-06-29|   CUST0019|Wireless Mouse|Electronics|          SP06|     Arjun Patel| South|       5|    929.24|          20|         929.24|     3716.96|    Debit Card|     Pend

In [16]:
df_nulls_fixed = df_nulls_fixed \
    .withColumn(
        "total_amount",
        when(
            col("total_amount").isNull(),
            spark_round(
                col("unit_price") * col("quantity") - col("discount_amount"), 2
            )
        ).otherwise(col("total_amount"))
    )

In [17]:
null_counts_fixed = df_nulls_fixed.select([ count(when(col(c).isNull(), c)).alias(c) for c in df_nulls_fixed.columns ]) 
null_counts_fixed.show(truncate=False) 

+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|product_id|order_id|order_date|customer_id|product_name|category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|0         |0       |0         |0          |0           |0       |0             |0               |0     |0       |0         |0           |0              |0           |0             |0           |
+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+



In [18]:
from pyspark.sql.functions import count as cnt

In [19]:
total    = df_nulls_fixed.count()
distinct = df_nulls_fixed.distinct().count()
print(f"Total rows       : {total:,}")
print(f"Distinct rows    : {distinct:,}")
print(f"Exact duplicates : {total - distinct:,}")

Total rows       : 325
Distinct rows    : 317
Exact duplicates : 8


In [20]:
df_nulls_fixed.groupBy(df_nulls_fixed.columns) \
  .count() \
  .filter(col("count") > 1) \
  .show()

+----------+--------+----------+-----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+-----+
|product_id|order_id|order_date|customer_id|  product_name|   category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|count|
+----------+--------+----------+-----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+-----+
|      P009| ORD1035|2024-05-02|   CUST0034|    Headphones|Electronics|          SP03|    Ananya Singh|  East|       4|   3174.31|          10|        1269.72|    11427.52|           UPI|    Returned|    2|
|      P002| ORD1022|2024-10-10|   CUST0029|    Smartphone|Electronics|          SP02|     Rahul Verma| South|       2|  18380.86|           0|            0.0|    29409.38|

In [21]:
df_nulls_fixed.filter(df_nulls_fixed.order_id =="ORD1035").show() 

+----------+--------+----------+-----------+------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|product_id|order_id|order_date|customer_id|product_name|   category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+----------+--------+----------+-----------+------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|      P009| ORD1035|2024-05-02|   CUST0034|  Headphones|Electronics|          SP03|    Ananya Singh|  East|       4|   3174.31|          10|        1269.72|    11427.52|           UPI|    Returned|
|      P009| ORD1035|2024-05-02|   CUST0034|  Headphones|Electronics|          SP03|    Ananya Singh|  East|       4|   3174.31|          10|        1269.72|    11427.52|           UPI|    Returned|
+----

In [22]:
df_nulls_fixed.filter(df_nulls_fixed.order_id =="ORD1039").show() 

+----------+--------+----------+-----------+------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|product_id|order_id|order_date|customer_id|product_name|   category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+----------+--------+----------+-----------+------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|      P009| ORD1039|2024-03-09|   CUST0071|  Headphones|Electronics|          SP01| Online / No Rep| North|       5|   3829.39|           0|            0.0|    19146.95|           UPI|   Completed|
|      P009| ORD1039|2024-03-09|   CUST0071|  Headphones|Electronics|          SP01| Online / No Rep| North|       5|   3829.39|           0|            0.0|    19146.95|           UPI|   Completed|
+----

In [23]:
df_dup_filxed=df_nulls_fixed.dropDuplicates()

In [24]:
df_dup_filxed.groupBy(df_nulls_fixed.columns) \
  .count() \
  .filter(col("count") > 1) \
  .show()

+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+-----+
|product_id|order_id|order_date|customer_id|product_name|category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|count|
+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+-----+
+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+-----+

